## Creación de dbf1 con chroma y llm

## Ojo: esta es la copia del código usado para crear dbf1. Su ejecución toma aprox 5 hrs y fue interrumpida y eventualmente termianda. No se recomienda su uso.

In [1]:
from dotenv import load_dotenv
import os

from IPython.display import HTML, Markdown, display
import pandas as pd
import openai

from openai import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

import tiktoken
import tqdm
import re
import ast


# Add this line to load variables from .env file
load_dotenv()

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')  # Retrieve the API key
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')  # Retrieve the API key
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')  # Retrieve the API key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


client = OpenAI()

client.embeddings.create(
  model='text-embedding-3-large', #"text-embedding-ada-002",
  input="The food was delicious and the waiter...",
  encoding_format="float"
)

embedding_fun_openai = OpenAIEmbeddingFunction(api_key=os.environ.get('OPENAI_API_KEY'), model_name="text-embedding-ada-002")

In [2]:
! git branch

* db_f1
  master
  test_enc_1
  test_enc_2


In [3]:
# carga de pickle con datos de encuestas
import pickle

ruta_enc= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/encuestas'

with open(f'{ruta_enc}/los_mex_dict.pkl', 'rb') as f:
    los_mex_dict = pickle.load(f)
    print('los_mex_dict cargado --  leer readme_dict para info')


enc_dict = los_mex_dict['enc_dict']
enc_nom_dict = los_mex_dict['enc_nom_dict']
pregs_dict = los_mex_dict['pregs_dict']
ses_dict = los_mex_dict['ses_dict']


los_mex_dict cargado --  leer readme_dict para info


### generación de dbf1 con todas las variables del proyecto

In [4]:
# tmp_var_dict ahora tendrá todas las variables de todo el proyecto

rev_enc_nom_dict = {v: k for k, v in enc_nom_dict.items()}
tmp_pregs_dict = pregs_dict.copy()
# remover Ponds que se hayan colado

tmp_pregs_dict = {ky: val for ky, val in pregs_dict.items()
    if not (ky.split('|')[0].startswith('Pond') or ky.split('|')[0].startswith('pond'))}


tmp_var_dict = {}


for k, v in tmp_pregs_dict.items():
    p_id, t_id = k.split('|')
    #print(p_id, t_id)
    df = enc_dict[rev_enc_nom_dict[t_id]]['dataframe']
    tmp_cols = df.filter(regex= 'Pond|pond').columns

    # ensure weight column is named 'Pondi2'
    if 'Pondi2' not in df.columns and 'pondi2' not in df.columns:
        raise KeyError(f"No 'Pondi2' or 'pondi2' column in dataframe for {t_id}")
    elif 'pondi2' in df.columns and 'Pondi2' not in df.columns:
        df.rename(columns={'pondi2': 'Pondi2'}, inplace=True)
    else: # fallback total si no da con el ponderador
        df['Pondi2'] = 1


    tmp_var = enc_dict[rev_enc_nom_dict[t_id]]['dataframe'][[p_id, 'Pondi2']].copy()

    tmp_var.columns = [p_id, 'ponderador']
    tmp_var['ponderador'] = tmp_var['ponderador'].astype(float)
    tmp_var['ponderador'] = tmp_var['ponderador'].fillna(0)
    tmp_var['ponderador'] = tmp_var['ponderador'] / tmp_var['ponderador'].sum()
    tmp_var.index.name = v
    tmp_var_dict[k] = tmp_var

# tmp_df_dict contiene los dataframes de cada variable

tmp_val_etq_dict = {}
tmp_df_dict = {}

def calculate_weighted_proportion(df, categorical_var, weight_var='ponderador', normalize=True):
    """
    Calculate the weighted proportion of a categorical variable.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The DataFrame containing the data
    categorical_var : str
        The name of the categorical variable column
    weight_var : str, default 'ponderador'
        The name of the weight variable column
    normalize : bool, default True
        If True, normalize weights to sum to 1
        
    Returns:
    --------
    pandas.DataFrame
        A DataFrame with categories as index and weighted proportions as values
    """
    # Ensure the weight variable is numeric
    df = df.copy()
    df[weight_var] = df[weight_var].astype(float)
    
    # Fill missing weights with 0
    df[weight_var] = df[weight_var].fillna(0)
    
    # Normalize weights if requested
    if normalize:
        df[weight_var] = df[weight_var] / df[weight_var].sum()
    
    # Group by the categorical variable and sum the weights
    weighted_counts = df.groupby(categorical_var)[weight_var].sum()
    
    # Create a DataFrame with the results
    result = weighted_counts.to_frame(name='proportion')
    
    return result

for tmp_ky in tmp_var_dict.keys():
    p_id, t_id = tmp_ky.split('|')
    
    # extrae y limpia etiquetas de variables
    print(f'tmp_ky = {tmp_ky}')

    print(f'p_id = {p_id}')
    print(f't_id = {t_id}')
    tmp_etq_dict = enc_dict[rev_enc_nom_dict[t_id]]['metadata']['variable_value_labels'].copy()
    tmp_etq_dict= {k.replace('a_1','').replace('a', ''): v for k, v in tmp_etq_dict.items()}

    if p_id not in tmp_etq_dict.keys():
        print(f'No se encuentra {p_id} en {tmp_etq_dict.keys()}')
        if p_id not in tmp_etq_dict.keys():
            print(f'No se encuentra {p_id} en {tmp_etq_dict.keys()}')
            continue

    tmp_etq = tmp_etq_dict[p_id].copy()
    tmp_etq = {k: v.replace('(esp)', '') for k, v in tmp_etq.items()}
    tmp_etq = {k: v.strip() for k, v in tmp_etq.items()}

    #limpia 8 o 98
    max_key = max(tmp_etq.keys())

    if max_key == 99.0:
        tmp_etq[max_key] = 'No sabe/ No contesta'
        tmp_etq.pop(98.0, None)
    elif max_key == 9.0:
        tmp_etq[max_key] = 'No sabe/ No contesta'
        tmp_etq.pop(8.0, None)

    tmp_val_etq_dict[tmp_ky] = tmp_etq

    # calcula y ajusta totales ponderados

    # tmp_pct_df =tmp_var_dict[tmp_ky].iloc[:, 0].value_counts(normalize=True).to_frame(name='%')
    tmp_pct_df = calculate_weighted_proportion(tmp_var_dict[tmp_ky], p_id, weight_var='ponderador', normalize=True)
    tmp_pct_df = tmp_pct_df.rename(columns={'proportion': '%'})
    # tmp_pnd_df = tmp_var_dict[tmp_ky].groupby(p_id)['ponderador'].sum()

    # tmp_pct_df = tmp_pct_df.mul(tmp_pnd_df, axis = 0)

    if max(tmp_pct_df.index) == 99.0:
        if 98.0 in tmp_pct_df.index:
            tmp_pct_df.loc[99.0] += tmp_pct_df.loc[98.0]
            tmp_pct_df = tmp_pct_df.drop(98.0)

    elif max(tmp_pct_df.index) == 9.0:
        if 8.0 in tmp_pct_df.index:
            tmp_pct_df.loc[9.0] += tmp_pct_df.loc[8.0]
            tmp_pct_df = tmp_pct_df.drop(8.0)
            
            

    # Use the values of the subdict as index
    tmp_pct_df.index = tmp_pct_df.index.map(tmp_val_etq_dict[tmp_ky])
    # print(tmp_val_etq_dict[tmp_ky])
    
    tmp_pct_df.index.name = tmp_var_dict[tmp_ky].index.name

    # print(tmp_pct_df)    


    # Store the dataframe in the dictionary
    tmp_df_dict[tmp_ky] = tmp_pct_df.mul(100).round(2)



def dataframe_to_markdown(df):
    """
    Converts a pandas DataFrame to a markdown table string with index name and values.
    
    Parameters:
        df (pd.DataFrame): The DataFrame to convert.
        
    Returns:
        str: A markdown table string representation of the DataFrame including index.
    """
    # Get column headers
    headers = df.columns.tolist()
    
    # Get index name (use empty string if None)
    index_name = df.index.name if df.index.name is not None else ""
    
    # Create header row with index name
    header_row = "| " + str(index_name) + " | " + " | ".join(str(col) for col in headers) + " |"
    
    # Create separator row (defines alignment)
    separator_row = "| --- | " + " | ".join(["---" for _ in headers]) + " |"
    
    # Create data rows with index values
    data_rows = []
    for idx, row in df.iterrows():
        formatted_values = []
        for val in row:
            if isinstance(val, float):
                formatted_values.append(f"{val:.2f}")
            else:
                formatted_values.append(str(val))
        
        # Include the index value as first column
        data_rows.append("| " + str(idx) + " | " + " | ".join(formatted_values) + " |")
    
    # Combine all parts
    markdown_table = "\n".join([header_row, separator_row] + data_rows)
    
    return markdown_table

tmp_markdown_tables = {key: dataframe_to_markdown(df) for key, df in tmp_df_dict.items()}
tmp_markdown_tables

tmp_ky = p1_1|IDE
p_id = p1_1
t_id = IDE
tmp_ky = p2_1|IDE
p_id = p2_1
t_id = IDE
tmp_ky = p3_1|IDE
p_id = p3_1
t_id = IDE
tmp_ky = p3_2|IDE
p_id = p3_2
t_id = IDE
tmp_ky = p3_3|IDE
p_id = p3_3
t_id = IDE
tmp_ky = p3_4|IDE
p_id = p3_4
t_id = IDE
tmp_ky = p3_5|IDE
p_id = p3_5
t_id = IDE
tmp_ky = p3_6|IDE
p_id = p3_6
t_id = IDE
tmp_ky = p4|IDE
p_id = p4
t_id = IDE
tmp_ky = p5_1|IDE
p_id = p5_1
t_id = IDE
tmp_ky = p6|IDE
p_id = p6
t_id = IDE
tmp_ky = p7|IDE
p_id = p7
t_id = IDE
tmp_ky = p8|IDE
p_id = p8
t_id = IDE
tmp_ky = p9|IDE
p_id = p9
t_id = IDE
tmp_ky = p10_1|IDE
p_id = p10_1
t_id = IDE
tmp_ky = p10_2|IDE
p_id = p10_2
t_id = IDE
tmp_ky = p10_3|IDE
p_id = p10_3
t_id = IDE
tmp_ky = p10_4|IDE
p_id = p10_4
t_id = IDE
tmp_ky = p10_5|IDE
p_id = p10_5
t_id = IDE
tmp_ky = p10_6|IDE
p_id = p10_6
t_id = IDE
tmp_ky = p10_7|IDE
p_id = p10_7
t_id = IDE
tmp_ky = p10_8|IDE
p_id = p10_8
t_id = IDE
tmp_ky = p11|IDE
p_id = p11
t_id = IDE
tmp_ky = p12|IDE
p_id = p12
t_id = IDE
tmp_ky = p13|IDE
p_id = 

{'p1_1|IDE': '| IDENTIDAD_Y_VALORES|Con la palabra maíz, yo asocio comida, mercado, animales. Dígame por favor, tres palabras que asocies con la palabra MÉXICO. 1° MENCIÓN | % |\n| --- | --- |\n| nan | 6.83 |\n| nan | 2.92 |\n| nan | 0.42 |\n| nan | 0.17 |\n| nan | 0.17 |\n| nan | 0.08 |\n| nan | 0.17 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 1.75 |\n| nan | 0.25 |\n| nan | 0.17 |\n| nan | 0.08 |\n| nan | 0.83 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.17 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.33 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.25 |\n| nan | 0.33 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.17 |\n| nan | 0.08 |\n| nan | 0.17 |\n| nan | 0.08 |\n| nan | 0.83 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.17 |\n| nan | 0.08 |\n| nan | 0.42 |\n| nan | 2.33 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 0.08 |\n| nan | 4.00 |\n| nan | 0.42 |\n| nan | 0.42 |\n| nan 

In [6]:
len(tmp_markdown_tables)

4212

In [ ]:
# carga de pickle con datos de encuestas
# save in los_mex_dict and add it to los_mex_dict.pkl
los_mex_dict['mkdown_tables'] = tmp_markdown_tables
with open(f'{ruta_enc}/los_mex_dict.pkl', 'wb') as f:
    pickle.dump(los_mex_dict, f)
    print('los_mex_dict.pkl actualizado con mkdown_tables')


los_mex_dict.pkl actualizado con mkdown_tables


In [ ]:
# generación de prompt para seleccionar analizar las tablas generadas . v2. SIN query.
# generación aumentada: prompt + pregunta + documentos 

tmp_nsnc_val = '2'
tmp_topic = None


def create_prompt_sum2(table_key, tmp_markdown_tables):
    """
    Creates a prompt for analyzing a table and answering a query.

    Parameters:
        query (str): The query to be answered.
        table_key (str): The key of the table to be analyzed.
        tmp_markdown_tables (dict): A dictionary containing markdown tables.
        tmp_nsnc_val (str): The threshold percentage for 'No sabe/ No contesta'.

    Returns:
        str: The generated prompt.
    """

    tmp_tab_st = tmp_markdown_tables[table_key]
    tmp_topic = tmp_tab_st.split('|')[1].replace('_', ' ').lower().strip()

    # subtasks
    subtask_str_1 = f"""
    write a short summary of the table; note that if the 'no answer/ does not know' option is present, you should include it in the summary if it too high (higher than {tmp_nsnc_val}%) and explain what it means. Otherwise, do not mention it. 
    you will include the percentages of all the options you discuss in this summary, with format (dd.d%) where d is a digit and the number represents the percentage of respondents that selected that option.
    """

    subtask_str_2 = f"""
    you will 1) think about the table you just described and answer the question: "Why is this table important for me as an expert in TOPIC "{tmp_topic}"? You will write this reason in a short sentence.
    And then you will 2) think how other experts in TOPIC "{tmp_topic}" could use these results and you new insights about their importance, and write a short paragraph about it. 

    If at first the relationship between the table and the TOPIC "{tmp_topic}" is not clear, you should recall that these questions are part of a larger questionnaire about TOPIC "{tmp_topic}". 
    This means that respondents knew that they were answering questions about TOPIC "{tmp_topic}", and that they were asked to answer these questions in the context of TOPIC "{tmp_topic}".
    If you think the table is not relevant to the TOPIC, do NOT say so BUT make an effort to imagine how they may be related and provide an answer.
    """

    # general prompt
    prompt = f"""
    You are an expert quantitative analyst who speaks English and Spanish fluently, and can comprehend any combination of the two languages. 
    You are also an expert in TOPIC "{tmp_topic}". If you do not understand what  "{tmp_topic}" is, make sure to translate it to English for your own work. 
    You will return your answer in English, but you can use Spanish to help you understand the topic.
    You are an expert in TOPIC "{tmp_topic}" capable of discussing quantitative results like tables and qualitative topics like opinions and values about it. 

    Your task is to analyze a TABLE containing the results of an opinion survey. 
    Note that the HEADER of the TABLE contains the question, the FIRST COLUMN contains the answer options, and the SECOND COLUMN contains the percentage of people that selected that answered each option.
    This is the TABLE: {tmp_tab_st}
    
    In your analysis you will: A) {subtask_str_1}, and B) {subtask_str_2}

    You will return a python dict with the following keys: 'summary' for tast A and 'implication' for task B.
    The values of the dict should be the answers to each task.
    IMPORTANT: The dict should be formatted as follows: {{'summary': '...', 'implication': '...'}}. 
    Make sure to return only correctly formatted python dict, without any code block formatting, markdown, or additional text.

    """
    return prompt

# Example usage:
prompt = create_prompt_sum2('p46_4|IDE', tmp_markdown_tables)


print(prompt)


    You are an expert quantitative analyst who speaks English and Spanish fluently, and can comprehend any combination of the two languages. 
    You are also an expert in TOPIC "identidad y valores". If you do not understand what  "identidad y valores" is, make sure to translate it to English for your own work. 
    You will return your answer in English, but you can use Spanish to help you understand the topic.
    You are an expert in TOPIC "identidad y valores" capable of discussing quantitative results like tables and qualitative topics like opinions and values about it. 

    Your task is to analyze a TABLE containing the results of an opinion survey. 
    Note that the HEADER of the TABLE contains the question, the FIRST COLUMN contains the answer options, and the SECOND COLUMN contains the percentage of people that selected that answered each option.
    This is the TABLE: | IDENTIDAD_Y_VALORES|Y Ahora dígame; en su opinión ¿qué tanto los mexicanos mienten para obtener un bene

In [ ]:
# Prepare the document list ensuring they are valid strings
docs = [ str(doc).strip() for doc in pregs_dict.values() if doc is not None and str(doc).strip() != "" ]

# Make sure the ids list is in sync with docs – for example, filter the keys too:
ids = [ key for key, doc in pregs_dict.items() if doc is not None and str(doc).strip() != "" ]


def get_answer(prompt, system_prompt=None, model='gpt-4o-mini-2024-07-18'):
    """Get a simple answer from OpenAI models without chatbot functionality."""
    from openai import OpenAI
    
    client = OpenAI(
        api_key=os.environ.get("OPENAI_API_KEY"),
    )
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    
    messages.append({"role": "user", "content": prompt})
    
    response = client.chat.completions.create(
        model=model, #'gpt-4.1-nano-2025-04-14', #"gpt-4o",
        messages=messages
    )
    
    return response.choices[0].message.content



def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Update the token count calculation in batch_documents to use num_tokens_from_string
# 8192 is the token limit for the model
def batch_documents(docs, ids, max_tokens=8192, encoding_name="cl100k_base"):
    """
    Batches documents and their corresponding IDs while respecting the token limit.

    Parameters:
        docs: List of documents to batch.
        ids: List of IDs corresponding to the documents.
        max_tokens: Maximum token limit for each batch.
        encoding_name: Encoding name for token calculation.

    Returns:
        List of batches, where each batch is a tuple of (batched_docs, batched_ids).
    """
    batches = []
    current_batch_docs = []
    current_batch_ids = []
    current_token_count = 0

    for doc, doc_id in zip(docs, ids):
        doc_token_count = num_tokens_from_string(doc, encoding_name)  # Use the token counting function
        
        print(f"Document ID: {doc_id}, Token Count: {doc_token_count}")
        
        if current_token_count + doc_token_count > max_tokens:
            # Save the current batch and start a new one
            batches.append((current_batch_docs, current_batch_ids))
            current_batch_docs = []
            current_batch_ids = []
            current_token_count = 0

        # Add the document to the current batch
        current_batch_docs.append(doc)
        current_batch_ids.append(doc_id)
        current_token_count += doc_token_count

    # Add the last batch if it has any documents
    if current_batch_docs:
        batches.append((current_batch_docs, current_batch_ids))

    return batches


# Create batches of documents
# batch_documents(docs, ids, max_tokens=8192, encoding_name="cl100k_base")



In [ ]:

tmp_pmt_lst= [create_prompt_sum2(ky_st, tmp_markdown_tables) for ky_st in tmp_markdown_tables.keys()]

tmp_docs= tmp_pmt_lst
tmp_ids = tmp_markdown_tables.keys()
tmp_bch_list = batch_documents(tmp_docs, tmp_ids, max_tokens=8192, encoding_name="cl100k_base")



answers_sum = {}
for batch_docs, batch_keys in tmp_bch_list:
    for prompt, key in zip(batch_docs, batch_keys):
        answers_sum[key] = get_answer(
            prompt=prompt,
            system_prompt=None,
            model='gpt-4.1-nano-2025-04-14'
        )
        
answers_sum



In [ ]:
# Resume interrupted LLM calls by reusing partial answers and processing the rest

# 1. Copy what we have so far
answers_partial = answers_sum.copy()

# 2. Identify which keys remain unprocessed
keys = list(tmp_markdown_tables.keys())
prompts = tmp_pmt_lst
remaining_indices = [i for i, k in enumerate(keys) if k not in answers_partial]
remaining_keys = [keys[i] for i in remaining_indices]
remaining_prompts = [prompts[i] for i in remaining_indices]

# 3. Batch the remaining prompts
batches_rem = batch_documents(remaining_prompts, remaining_keys,
                              max_tokens=8192, encoding_name="cl100k_base")

# 4. Count total remaining items for the progress bar
total_rem = sum(len(batch_keys) for _, batch_keys in batches_rem)

# 5. Call the LLM for each batch with a tqdm bar
with tqdm(total=total_rem, desc="Resuming LLM calls") as pbar:
    for batch_docs, batch_keys in batches_rem:
        for prompt, key in zip(batch_docs, batch_keys):
            answers_partial[key] = get_answer(prompt=prompt)
            pbar.update(1)

# 6. Combine into the final dictionary
answers_sum = answers_partial

Resuming LLM calls: 100%|██████████| 2586/2586 [2:37:36<00:00,  3.66s/it]  


In [ ]:
import os, pickle

dirname = ruta_enc            # e.g. "/path/to/output/dir"
filename = "answers_sum.pkl"
fullpath = os.path.join(dirname, filename)

os.makedirs(dirname, exist_ok=True)

with open(fullpath, 'wb') as f:
    pickle.dump(answers_sum, f)
 

In [ ]:
answers_sum['p1_1|IDE']

'{\'summary\': "The table presents responses to the question: \'Please give three words you associate with Mexico,\' with the answer options not explicitly listed but represented as \'nan\' (no answer). The percentages vary widely, with some high values such as 12.83%, 4.42%, and 4.00%, indicating notable proportions of respondents selecting those options. Several smaller percentages range from about 0.08% to around 2.5%, reflecting diverse associations or lack of responses. The high \'no answer\' or \'did not know\' options (some over 12%) suggest a significant portion of participants either did not respond or were uncertain, which may imply a divergence in perceptions or a lack of strong associations with Mexico.","implication": "Understanding these associations is crucial for exploring Mexican identity and values, as they reveal which concepts or symbols are most salient or meaningful to people. Other experts in \'identidad y valores\' can utilize these results to identify common cu

In [ ]:
import os
import pickle

# Path where you saved answers_sum.pkl
ruta_enc = '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/encuestas'
filename = 'answers_sum.pkl'
fullpath = os.path.join(ruta_enc, filename)

# Unpickle
with open(fullpath, 'rb') as f:
    answers_sum = pickle.load(f)

# Quick check
print(f"Loaded {len(answers_sum)} entries:")
print(list(answers_sum.keys())[:10])

Loaded 4212 entries:
['p1_1|IDE', 'p2_1|IDE', 'p3_1|IDE', 'p3_2|IDE', 'p3_3|IDE', 'p3_4|IDE', 'p3_5|IDE', 'p3_6|IDE', 'p4|IDE', 'p5_1|IDE']


In [ ]:
import ast

answers_dict = {}
for key, resp_text in answers_sum.items():
    try:
        # try to parse the response as a Python literal (dict)
        answers_dict[key] = ast.literal_eval(resp_text)
    except (ValueError, SyntaxError):
        # if it fails, mark as incorrect
        answers_dict[key] = "incorrect"

# inspect the result
answers_dict

{'p1_1|IDE': {'summary': "The table presents responses to the question: 'Please give three words you associate with Mexico,' with the answer options not explicitly listed but represented as 'nan' (no answer). The percentages vary widely, with some high values such as 12.83%, 4.42%, and 4.00%, indicating notable proportions of respondents selecting those options. Several smaller percentages range from about 0.08% to around 2.5%, reflecting diverse associations or lack of responses. The high 'no answer' or 'did not know' options (some over 12%) suggest a significant portion of participants either did not respond or were uncertain, which may imply a divergence in perceptions or a lack of strong associations with Mexico.",
  'implication': "Understanding these associations is crucial for exploring Mexican identity and values, as they reveal which concepts or symbols are most salient or meaningful to people. Other experts in 'identidad y valores' can utilize these results to identify common

In [ ]:
from tqdm import tqdm
import chromadb
from chromadb import PersistentClient                             # CHANGED
from chromadb.config import Settings, DEFAULT_TENANT, DEFAULT_DATABASE  # CHANGED
import tiktoken

# --------------------------------------------------------------------
# 1) Configure your persistent client (v0.4+)
# --------------------------------------------------------------------
DB_PERSIST_DIR = "./db_f1"
DB_NAME        = "enc_dbf1"
reset_db       = False

client = PersistentClient(
    path     = DB_PERSIST_DIR,
    settings = Settings(allow_reset=True),   # allow resetting on-disk state
    tenant   = DEFAULT_TENANT,
    database = DEFAULT_DATABASE,
)  # :contentReference[oaicite:0]{index=0}

# --------------------------------------------------------------------
# 2) Create or reset the collection
# --------------------------------------------------------------------
if reset_db:
    # deletes the whole collection directory if it exists
    try:
        client.delete_collection(name=DB_NAME)     # :contentReference[oaicite:1]{index=1}
    except ValueError:
        pass

db_f1 = client.get_or_create_collection(
    name               = DB_NAME,
    embedding_function = embedding_fun_openai
)

# --------------------------------------------------------------------
# 3) Build your 3-facet docs (question, summary, implications)
# --------------------------------------------------------------------
docs, ids, metadatas = [], [], []
for qid in pregs_dict:
    q_text = str(pregs_dict[qid]).strip()

    if answers_dict.get(qid, "") == "incorrect" or answers_dict.get(qid, "") == "":
        print(f"Skipping incorrect answer for {qid}")
        continue

    s_text = str(answers_dict.get(qid, "")['summary']).strip()
    i_text = str(answers_dict.get(qid, "")['implication']).strip()

    for facet, text in [
        ("question",     q_text),
        ("summary",      s_text),
        ("implications", i_text),
    ]:
        if text:
            docs.append(text)
            ids.append(f"{qid}__{facet}")
            metadatas.append({"qid": qid, "type": facet})

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    encoding = tiktoken.get_encoding(encoding_name)
    return len(encoding.encode(string))

def batch_documents(docs, ids, metadatas,
                    max_tokens=8192, encoding_name="cl100k_base"):
    batches = []
    cur_docs, cur_ids, cur_metas, cur_tokens = [], [], [], 0

    for doc, did, meta in zip(docs, ids, metadatas):
        ntoks = num_tokens_from_string(doc, encoding_name)
        if cur_tokens + ntoks > max_tokens:
            batches.append((cur_docs, cur_ids, cur_metas))
            cur_docs, cur_ids, cur_metas, cur_tokens = [], [], [], 0

        cur_docs.append(doc)
        cur_ids.append(did)
        cur_metas.append(meta)
        cur_tokens += ntoks

    if cur_docs:
        batches.append((cur_docs, cur_ids, cur_metas))
    return batches

# --------------------------------------------------------------------
# 4) Batch, embed (with inner progress bar), and add to db_f1
# --------------------------------------------------------------------
batches = batch_documents(
    docs, ids, metadatas,
    max_tokens=8192,
    encoding_name="cl100k_base"
)

for batch_docs, batch_ids, batch_metas in tqdm(batches, desc="Adding batches to DB"):
    embeddings = []
    for doc in tqdm(batch_docs, desc="  Embedding docs", leave=False):
        emb = embedding_fun_openai([doc])[0]              
        embeddings.append(emb)

    db_f1.add(
        documents = batch_docs,
        ids       = batch_ids,
        metadatas = batch_metas,
        embeddings= embeddings
    )


print("Total vectors:", db_f1.count())
print("Sample peek:", db_f1.peek())


Skipping incorrect answer for p36_5|IDE
Skipping incorrect answer for p46_2|IDE
Skipping incorrect answer for p51_2|IDE
Skipping incorrect answer for p62_1_o|IDE
Skipping incorrect answer for p66_1|IDE
Skipping incorrect answer for p3_3|MED
Skipping incorrect answer for p5|MED
Skipping incorrect answer for p10_6|MED
Skipping incorrect answer for p15_2|MED
Skipping incorrect answer for p30_2|MED
Skipping incorrect answer for p30_3|MED
Skipping incorrect answer for p33_6|MED
Skipping incorrect answer for p48_2g|MED
Skipping incorrect answer for p17_1|POB
Skipping incorrect answer for p18_10|POB
Skipping incorrect answer for p15_6|CUL
Skipping incorrect answer for p48_6|CUL
Skipping incorrect answer for p53_1|CUL
Skipping incorrect answer for p53_6|CUL
Skipping incorrect answer for p54_3|CUL
Skipping incorrect answer for p58_2|CUL
Skipping incorrect answer for p81_1|CUL
Skipping incorrect answer for p22_2|REL
Skipping incorrect answer for p33_11|REL
Skipping incorrect answer for p44_1|REL

Adding batches to DB: 100%|██████████| 111/111 [1:02:28<00:00, 33.77s/it]


AttributeError: 'Client' object has no attribute 'persist'

In [ ]:
print("Total vectors:", db_f1.count())


Total vectors: 11235


In [ ]:
print("Sample peek:", db_f1.peek())


Sample peek: {'ids': ['p1_1|IDE__question', 'p1_1|IDE__summary', 'p1_1|IDE__implications', 'p2_1|IDE__question', 'p2_1|IDE__summary', 'p2_1|IDE__implications', 'p3_1|IDE__question', 'p3_1|IDE__summary', 'p3_1|IDE__implications', 'p3_2|IDE__question'], 'embeddings': array([[-0.01286233, -0.02477617,  0.00845953, ..., -0.00216455,
        -0.01462473, -0.00672276],
       [-0.00439202, -0.005377  ,  0.03002077, ..., -0.00975933,
        -0.02509912, -0.04386856],
       [-0.03001864,  0.00425975,  0.03407801, ...,  0.00466375,
        -0.01384581, -0.03074261],
       ...,
       [ 0.02034639,  0.01068023,  0.01885129, ..., -0.00335098,
        -0.01566607, -0.04771326],
       [-0.01296452,  0.00133349,  0.02093472, ..., -0.00910233,
        -0.00846187, -0.02445403],
       [-0.00732108, -0.0118375 ,  0.03220749, ..., -0.00837447,
        -0.01164658, -0.00516163]], shape=(10, 1536)), 'documents': ['IDENTIDAD_Y_VALORES|Con la palabra maíz, yo asocio comida, mercado, animales. Dígame po